**Problem 9**

The state wildlife biologists want to model how many fish are being caught by fishermen at a state park. Visitors are asked how long they stayed, how many people were in the group, whether there were children in the group, and how many fish were caught. Some visitors do not fish, but there is no data on whether a person fished or not. Some visitors who did fish did not catch any fish, so there are excess zeros in the data because of the people that did not fish. The dataset is taken from:

UCLA Fish Dataset

Fit the Zero-Inflated Negative Binomial (ZINB) regression generalized linear model (GLM) to identify the factors associated with the number of fish caught. Interpret the results.

In [1]:
import pandas as pd
import numpy as np

import statsmodels.api as sm

from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

In [17]:
url = "https://stats.idre.ucla.edu/stat/data/fish.csv"

df = pd.read_csv(url)

In [18]:
df.head()

,nofish,livebait,camper,persons,child,xb,zg,count
0,1,0,0,1,0,-0.896315,3.050405,0
1,0,1,1,1,0,-0.558345,1.746149,0
2,0,1,0,1,0,-0.401731,0.279939,0
3,0,1,1,2,1,-0.956298,-0.601526,0
4,0,1,0,1,0,0.436891,0.527709,1


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   nofish    250 non-null    int64  
 1   livebait  250 non-null    int64  
 2   camper    250 non-null    int64  
 3   persons   250 non-null    int64  
 4   child     250 non-null    int64  
 5   xb        250 non-null    float64
 6   zg        250 non-null    float64
 7   count     250 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 15.8 KB


In [20]:
df.describe()

,nofish,livebait,camper,persons,child,xb,zg,count
count,250.000000,250.000000,250.000000,250.00000,250.000000,250.000000,250.000000,250.000000
mean,0.296000,0.864000,0.588000,2.52800,0.684000,0.973796,0.252323,3.296000
std,0.457407,0.343476,0.493182,1.11273,0.850315,1.440277,2.102391,11.635028
min,0.000000,0.000000,0.000000,1.00000,0.000000,-3.275050,-5.625944,0.000000
25%,0.000000,1.000000,0.000000,2.00000,0.000000,0.008267,-1.252724,0.000000
50%,0.000000,1.000000,1.000000,2.00000,0.000000,0.954550,0.605079,0.000000
75%,1.000000,1.000000,1.000000,4.00000,1.000000,1.963855,1.993237,2.000000
max,1.000000,1.000000,1.000000,4.00000,3.000000,5.352674,4.263185,149.000000


In [21]:
#Check Excess Zeros
zeros = (df['count'] == 0).sum()

print("Number of Zeros:", zeros)

print(
    "Percentage of Zeros:",
    zeros / len(df) * 100
)

Number of Zeros: 142
Percentage of Zeros: 56.8


In [22]:
#Mean and Variance
print(
    "Mean:",
    df['count'].mean()
)

print(
    "Variance:",
    df['count'].var()
)

Mean: 3.296
Variance: 135.37387951807236


since Variance >> Mean
so  overdispersion exists

This supports using ZINB instead of ZIP.

In [23]:
#Prepare Variables

#Response variable:

y = df['count']

In [24]:
#Predictors:

X = df[['child', 'persons', 'camper']]

In [25]:
X = sm.add_constant(X)

In [15]:
#Fit ZINB Model

#Model:count∼child+persons+camper
zinb_model = ZeroInflatedNegativeBinomialP(
    endog=y,
    exog=X,
    exog_infl=X,
    inflation='logit'
).fit()

         Current function value: 1.620380
         Iterations: 35
         Function evaluations: 37
         Gradient evaluations: 37


/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3937: RuntimeWarning: invalid value encountered in log
  a1 * np.log(a1) + y * np.log(mu) -
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3938: RuntimeWarning: invalid value encountered in log
  (y + a1) * np.log(a2))
/usr/local/lib/python3.12/dist-packages/scipy/optimize/_optimize.py:1330: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [26]:
print(zinb_model.summary())

                     ZeroInflatedNegativeBinomialP Regression Results                    
Dep. Variable:                             count   No. Observations:                  250
Model:             ZeroInflatedNegativeBinomialP   Df Residuals:                      246
Method:                                      MLE   Df Model:                            3
Date:                           Sun, 10 May 2026   Pseudo R-squ.:                  0.1278
Time:                                   12:40:17   Log-Likelihood:                -405.09
converged:                                 False   LL-Null:                       -464.44
Covariance Type:                       nonrobust   LLR p-value:                 1.479e-25
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
inflate_const      -0.4892        nan        nan        nan         nan         nan
inflate_child       0.0378  

**Interpretation of Zero-Inflated Negative Binomial (ZINB) Results**

The Zero-Inflated Negative Binomial model was fitted to analyze fish catch counts while accounting for:

excess zeros
overdispersion in the data

The overall model is statistically significant:

Likelihood Ratio p-value = 1.48 × 10⁻²⁵

This suggests that the predictors collectively help explain fish catch counts.